# verl 中文课程导航

本课程通过可运行的 Notebook 讲解 GPU 计算与 verl 训练。实验使用随机初始化的小模型，无需下载大模型权重。参考环境为 WSL、Python (verl GPU) 内核和 RTX 4060 8GB；其他设备的运行时间和资源占用可能不同。

**使用方法：** 点击课程链接，选择 Python (verl GPU)，从上到下按 Shift+Enter。仓库中的 Notebook 已清空输出，请按顺序运行以生成自己的结果；交互按钮需要先执行初始化代码。不要同时训练多个 Notebook；完成后用 Kernel → Shut Down Kernel 释放显存，关闭标签不一定关闭内核。

## 第一次碰到 verl 代码时怎么读

以实验 01 的这三行写法为例，先认出是谁在解释它，再预测它会产生什么：

| 你看到的写法 | 谁来解释 | 读完后试着回答 |
|---|---|---|
| `from verl import DataProto` | Python 从 verl 包导入批数据容器 | 这里只有导入。下一行传入两行 token ID，`len(batch)` 会是多少？ |
| `python -m verl.trainer.main_ppo --cfg job` | Python 找到模块，Hydra 读取配置 | 加上 `--cfg job` 后，输出应是配置树。去树里找 `algorithm` 这一层。 |
| `actor_rollout_ref.rollout.n=2` | Hydra 用右侧的值覆盖左侧配置项 | 从右往左读：`n` 是每题生成几条回答，`rollout` 是生成阶段。如果一批有 4 道题，共生成几条？ |

第三行的答案是 8 条，但 GRPO 比较的是 **4 个各含 2 条回答的小组**。给同一道题的两个回答都打 1 分，平均奖励是 1，组内优势却都是 0。实验 01 可以看到优势的正负号，实验 11 可以对照真实训练配置，实验 12 可以对照奖励和梯度曲线。

## 学习路线

| 课次 | 内容 | 学会什么 |
|---|---|---|
| [01](01_从GPU到verl的第一步.ipynb) | GPU 与奖励学习入门 | 设备、计时、梯度、奖励 |
| [02](02_训练微型语言模型.ipynb) | 微型 Qwen2 SFT | token、标签、生成、评估 |
| [03](03_用组内奖励微调语言模型.ipynb) | GRPO 风格教学循环 | 组采样、优势、裁剪、KL |
| [04](04_对比冻结模型、监督训练与奖励微调.ipynb) | 三组对照 | 冻结/SFT/RL、公平比较的边界 |
| [05](05_用交互评估台检验模型.ipynb) | 交互评估台 | 分布变化与能力边界 |
| [06](06_用数据覆盖改善泛化.ipynb) | 数据覆盖 | 分组隔离、验证与最终测试 |
| [07](07_逐位推理与长整数加法.ipynb) | 逐位组合 | 人工算法先验与组合泛化 |
| [08](08_采样温度与奖励漏洞.ipynb) | 采样和奖励漏洞 | 探索、多样性、奖励测试 |
| [09](09_保存检查点与恢复训练.ipynb) | 保存和恢复 | 权重、优化器与随机状态 |
| [10](10_将数据和模型接入verl.ipynb) | 真实 verl 接口与结业 | tokenizer、Parquet、奖励、配置 |

建议首次按顺序学习。已有 01 至 07 输出时，可以直接学习 08 至 10。每课通常阅读 10 至 25 分钟，计算时间以实际日志为准。

## 依赖与文件

- 02 → 03；04 建立新起点并保留旧结果。
- 04 → 05 → 06 → 07；08 使用 04；09 独立；10 使用 06 并检查课程产物。
- 各课输出在 `support/experimentXX_output/`，重跑保存格会覆盖对应课输出。
- `build_*.py` 是作者生成脚本，不是日常学习入口，运行会覆盖对应 Notebook 的修改和输出。
- 在 verl 项目根目录运行 `jupyter lab --notebook-dir=tutorial`，打开终端打印的本机链接。安装步骤见仓库 README。

## 常见问题

缺少变量：从初始化格重跑；错误内核：选 Python (verl GPU)；显存不足：关闭其他内核；旧警告：重启内核后重跑；按钮无响应：执行面板之前的代码格。进度条或按钮无法显示时，检查当前环境是否已安装 ipywidgets，并重启内核。

## 完成基础部分的标准

你应能独立解释 SFT 与奖励学习的差别、避免训练测试泄漏、判断真实改善、复现 checkpoint 恢复，并在第 10 课通过真实数据加载和配置检查。

**课程范围：** 第 03/04 课是 PyTorch 教学版 GRPO 风格实现；第 07 课是局部网络与显式竖式程序；第 10 课调用真实 verl 数据类和配置入口，但没有执行完整 Ray/vLLM 训练。完成基础课程不等同于已验证完整大模型 RL 训练。进阶阶段再做完整单步训练、显存优化和部署。


## 高级课程

从第 11 课开始验证真实 verl 训练，再把失败诊断推进到梯度、预热、评估和资源测量。

| 课程 | 核心问题 | 运行依赖 |
|---|---|---|
| [11：单步 GRPO](11_运行单步GRPO训练.ipynb) | 参数更新和检查点是否真实发生？ | 先运行配套脚本，或开启训练开关 |
| [12：稀疏奖励诊断](12_诊断稀疏奖励.ipynb) | 流程完成后，任务为何仍失败？ | 先运行 support/advanced12/run.sh |
| [13：目标函数与梯度](13_从目标函数推导策略梯度.ipynb) | 公式、mask 与梯度是否一致？ | CPU 即可，无前课权重依赖 |
| [14：SFT 预热与 verl](14_用监督微调预热GRPO.ipynb) | 预热能否改善协议与任务表现？ | CUDA；真实 GRPO 是单独启动的可选步骤 |
| [15：评估不确定性](15_估计评估结果的不确定性.ipynb) | 改善有多大，证据有多强？ | 第 14 课逐题预测，CPU 即可 |
| [16：显存与吞吐](16_测量显存与训练吞吐.ipynb) | 资源花在哪里，怎样正确计时？ | CUDA，无前课权重依赖 |
| [17：预训练模型 LoRA](17_用LoRA微调和导出预训练模型.ipynb) | 模板、掩码、适配器重载与合并是否可靠？ | CUDA、PEFT；首次下载约 1GB 权重 |
| [18：本地流式服务](18_测试流式推理服务与并发性能.ipynb) | 服务输出与并发指标是否可信？ | 第 17 课合并模型、vLLM；仅监听本机 |

第 11 课使用合成奖励验证工程链路。第 12 课的历史结果是 10 步中 3 步非零梯度，贪心测试准确率未改善。第 14 课更换词表和数据范围，属于独立实验，不与第 12 课直接作同条件因果比较。

首次克隆后没有本地模型、日志和检查点。请先执行相应训练；公开 Notebook 已清空输出，小型 JSON/CSV 是历史运行记录。每次训练以本轮配置与输出为准。

学习要求、缺口审阅和后续全量微调对照、多卡及生产负载项目见 [工程学习路线与考核](support/docs/ENGINEERING_ROADMAP.md)。


---

<a href="https://github.com/SergioTermann"><img src="support/assets/logo.png" alt="SergioTermann" width="44" height="44" align="left" style="margin: 0 12px 0 0;"></a>

**更多有趣的 AI 课程**  
[← 访问 SergioTermann 的 GitHub 主页](https://github.com/SergioTermann)